In [1]:
# ─────────────────────────────────────────────────────────
# Install all required packages
# langchain==1.3.0 | langchain-openai==1.2.1 | langchain-community==0.4.1
# ─────────────────────────────────────────────────────────
%pip install -q langchain==1.3.0
%pip install -q langchain-openai==1.2.1
%pip install -q langchain-community==0.4.1
%pip install -q faiss-cpu
%pip install -q openai
%pip install -q gradio
%pip install -q pypdf                  # PDF loading
%pip install -q python-docx            # Word doc loading
%pip install -q beautifulsoup4 lxml    # HTML loading
%pip install -q Pillow                 # Image handling
%pip install -q python-dotenv         # .env support
%pip install -q docx    
%pip install docx2txt
%pip install langchain-text-splitters
print("✅ All packages installed. Please restart the kernel before proceeding.")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ All packages installed. Please restart the kernel before proceeding.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# ── Load API key from .env file (recommended) OR set directly ──
load_dotenv()  # reads OPENAI_API_KEY from .env file in current directory

# If you don't have a .env file, uncomment and set directly:


OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("❌ OPENAI_API_KEY not found. Please set it in .env or directly above.")


In [3]:
SOURCE_FOLDER = Path("source_docs")       # Put your PDFs, DOCX, HTML here
print(SOURCE_FOLDER)

source_docs


In [ ]:
FAISS_INDEX_PATH = Path("faiss_index")    # FAISS vector store will be saved here

In [5]:
IMAGE_CACHE_PATH = Path("image_cache")    # Extracted images will be saved here

In [6]:
for folder in [SOURCE_FOLDER, FAISS_INDEX_PATH, IMAGE_CACHE_PATH]:
    folder.mkdir(exist_ok=True)

print(f"📁 Source folder: {SOURCE_FOLDER.resolve()}")
print(f"📁 FAISS index folder: {FAISS_INDEX_PATH.resolve()}")
print(f"📁 Image cache folder: {IMAGE_CACHE_PATH.resolve()}")
print()

📁 Source folder: D:\Training\Credo\github\gen-ai-repo\gen-ai-repo\module-5\source_docs
📁 FAISS index folder: D:\Training\Credo\github\gen-ai-repo\gen-ai-repo\module-5\faiss_index
📁 Image cache folder: D:\Training\Credo\github\gen-ai-repo\gen-ai-repo\module-5\image_cache



In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# ── LLM Wrapper: ChatOpenAI ──────────────────────────────────────────────
# model: gpt-4o-mini is cost-effective for a student chatbot
# temperature=0 → deterministic, factual answers (ideal for exam prep)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=OPENAI_API_KEY
)

In [8]:
# ── Vision LLM: GPT-4o for image understanding ────────────────────────────
# Used to summarize concept map images extracted from DOCX files
vision_llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    max_tokens=500,
    openai_api_key=OPENAI_API_KEY
)

In [9]:
# ── Quick Test: Direct LLM call ───────────────────────────────────────────
messages = [
    SystemMessage(content=(
        "You are a helpful tutor for 10th Standard Tamil Nadu State Board students. "
        "Answer in simple, clear language suitable for a 15-year-old student."
    )),
    HumanMessage(content="What is photosynthesis? Give a one-line definition.")
]


In [10]:
response = llm.invoke(messages)
print("🤖 LLM Direct Response (no RAG yet):")
print(response.content)
print()
print(f"📊 Model: {response.response_metadata.get('model_name', 'gpt-4o-mini')}")
print(f"📊 Input tokens: {response.usage_metadata.get('input_tokens', 'N/A')}")
print(f"📊 Output tokens: {response.usage_metadata.get('output_tokens', 'N/A')}")

🤖 LLM Direct Response (no RAG yet):
Photosynthesis is the process by which green plants use sunlight to convert carbon dioxide and water into glucose and oxygen.

📊 Model: gpt-4o-mini-2024-07-18
📊 Input tokens: 53
📊 Output tokens: 22


In [ ]:
print(response.usage_metadata) #langserve

{'input_tokens': 53, 'output_tokens': 22, 'total_tokens': 75, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [12]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    BSHTMLLoader
)
from langchain_core.documents import Document
import zipfile
import shutil
from PIL import Image
import io

In [13]:
all_documents = []   # will hold all loaded text Documents

In [14]:
pdf_files   = list(SOURCE_FOLDER.glob("*.pdf"))
docx_files  = list(SOURCE_FOLDER.glob("*.docx"))
html_files  = list(SOURCE_FOLDER.glob("*.html"))

In [15]:
print(pdf_files)

[WindowsPath('source_docs/science_chapter3.pdf')]


In [16]:
print(f"📄 Found {len(pdf_files)} PDF(s), {len(docx_files)} DOCX(s), {len(html_files)} HTML(s)")

📄 Found 1 PDF(s), 1 DOCX(s), 2 HTML(s)


In [17]:
# ── Load PDFs ────────────────────────────────────────────────────────────
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_type"] = "pdf"
        doc.metadata["file_name"]   = pdf_path.name
    all_documents.extend(docs)
    print(f"✅ PDF loaded: {pdf_path.name} → {len(docs)} page(s)")

✅ PDF loaded: science_chapter3.pdf → 14 page(s)


In [18]:
print(all_documents)

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2019-04-01T17:20:00+05:30', 'author': "BYJU'S", 'keywords': 'Tamilnadu Board Class 10 Science Chapter 12', 'moddate': '2019-04-01T17:20:30+05:30', 'subject': 'Tamilnadu Board Class 10 Science Chapter 12', 'title': 'Tamilnadu Board Class 10 Science Chapter 12', 'source': 'source_docs\\science_chapter3.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_type': 'pdf', 'file_name': 'science_chapter3.pdf'}, page_content='173\n Introduction \nPlants exhibits varying degrees of \norganization. Atoms are organized into \nmolecules, molecules into organelles, organelles \ninto cells, cells into tissues and tissues into \norgans. The first account of internal structure \nof plants was published by English Physician \nNehemiah Grew. He is known as Father of \nPlant Anatomy. Plant anatomy (Gk Ana = as \nunder; T emnein = to cut) is the study of internal \nstructure o

In [19]:
# ── Load DOCX (text) ─────────────────────────────────────────────────────
for docx_path in docx_files:
    loader = Docx2txtLoader(str(docx_path))
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_type"] = "docx"
        doc.metadata["file_name"]   = docx_path.name
    all_documents.extend(docs)
    print(f"✅ DOCX loaded: {docx_path.name} → {len(docs)} chunk(s)")

✅ DOCX loaded: history_notes.docx → 1 chunk(s)


In [20]:
# ── Load HTML ─────────────────────────────────────────────────────────────
for html_path in html_files:
    loader = BSHTMLLoader(str(html_path), open_encoding="utf-8")
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_type"] = "html"
        doc.metadata["file_name"]   = html_path.name
    all_documents.extend(docs)
    print(f"✅ HTML loaded: {html_path.name} → {len(docs)} page(s)")

✅ HTML loaded: maths_revision.html → 1 page(s)
✅ HTML loaded: science_model_question_paper.html → 1 page(s)


In [21]:
print(f"📚 Total raw documents loaded: {len(all_documents)}")
print()
print("🔍 Sample document preview:")
if all_documents:
    sample = all_documents[0]
    print(f"   Metadata: {sample.metadata}")
    print(f"   Content snippet: {sample.page_content[:200]}...")

📚 Total raw documents loaded: 17

🔍 Sample document preview:
   Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2019-04-01T17:20:00+05:30', 'author': "BYJU'S", 'keywords': 'Tamilnadu Board Class 10 Science Chapter 12', 'moddate': '2019-04-01T17:20:30+05:30', 'subject': 'Tamilnadu Board Class 10 Science Chapter 12', 'title': 'Tamilnadu Board Class 10 Science Chapter 12', 'source': 'source_docs\\science_chapter3.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_type': 'pdf', 'file_name': 'science_chapter3.pdf'}
   Content snippet: 173
 Introduction 
Plants exhibits varying degrees of 
organization. Atoms are organized into 
molecules, molecules into organelles, organelles 
into cells, cells into tissues and tissues into 
organs...


In [22]:
import base64
from langchain_core.messages import HumanMessage


def extract_images_from_docx(docx_path: Path, output_dir: Path) -> list:
    """Extract all images embedded in a .docx file."""
    extracted = []
    output_dir.mkdir(exist_ok=True)
    
    # DOCX files are ZIP archives — images are in word/media/
    with zipfile.ZipFile(str(docx_path), 'r') as z:
        media_files = [f for f in z.namelist() if f.startswith('word/media/')]
        for media_file in media_files:
            ext = Path(media_file).suffix.lower()
            if ext in ['.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff']:
                image_data = z.read(media_file)
                image_name = f"{docx_path.stem}_{Path(media_file).name}"
                save_path  = output_dir / image_name
                with open(save_path, 'wb') as f:
                    f.write(image_data)
                extracted.append({
                    "path": save_path,
                    "source_docx": docx_path.name,
                    "image_name": image_name
                })
    return extracted

In [23]:
def summarize_image_with_vision(image_path: Path, vision_llm, source_file: str) -> str:
    """Send image to GPT-4o Vision and get an educational summary."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    
    # Determine MIME type
    ext = image_path.suffix.lower().lstrip('.')
    mime_map = {'jpg': 'jpeg', 'jpeg': 'jpeg', 'png': 'png', 'gif': 'gif', 'bmp': 'bmp'}
    mime_type = f"image/{mime_map.get(ext, 'png')}"
    
    b64_image = base64.b64encode(image_bytes).decode('utf-8')
    
    message = HumanMessage(content=[
        {
            "type": "text",
            "text": (
                "You are analyzing an educational image from a 10th Standard Tamil Nadu State Board "
                "study material. This image is from the file: " + source_file + ".\n\n"
                "Please provide a detailed educational description of this image. Include:\n"
                "1. What concept or topic does this image represent?\n"
                "2. All labels, text, or annotations visible in the image\n"
                "3. The relationships shown (arrows, connections, hierarchy)\n"
                "4. How a student should interpret this for their exam\n"
                "Describe it as if explaining to a 10th std student who cannot see the image."
            )
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{b64_image}",
                "detail": "high"
            }
        }
    ])
    
    response = vision_llm.invoke([message])
    return response.content


In [24]:
# ── Process all DOCX files for images ────────────────────────────────────
image_documents = []  # Documents created from image summaries

In [25]:
for docx_path in docx_files:
    print(f"🔍 Scanning {docx_path.name} for embedded images...")
    images = extract_images_from_docx(docx_path, IMAGE_CACHE_PATH)
    
    if not images:
        print(f"   ℹ️  No images found in {docx_path.name} (sample file has no embedded images)")
        # Create a placeholder showing this capability works
        placeholder_doc = Document(
            page_content=(
                "[IMAGE DESCRIPTION PLACEHOLDER] This document section contains a concept map "
                "showing the digestive system flow: Mouth → Pharynx → Oesophagus → Stomach → "
                "Small Intestine (Duodenum, Jejunum, Ileum) → Large Intestine → Rectum → Anus. "
                "The image also shows associated glands: Salivary glands secrete saliva with amylase, "
                "Liver produces bile stored in gallbladder, Pancreas produces pancreatic juice "
                "containing lipase, trypsin, and amylase. "
                "This is from the teacher notes on the Human Digestive System."
            ),
            metadata={
                "source_type": "image_summary",
                "file_name": docx_path.name,
                "image_name": "digestive_system_concept_map",
                "description": "Placeholder - replace with real image summaries from GPT-4o Vision"
            }
        )
        image_documents.append(placeholder_doc)
        continue
    
    print(f"   🖼️  Found {len(images)} image(s). Sending to GPT-4o Vision...")
    for img_info in images:
        try:
            summary = summarize_image_with_vision(
                img_info["path"], vision_llm, img_info["source_docx"]
            )
            img_doc = Document(
                page_content=f"[IMAGE SUMMARY from {img_info['source_docx']}]: {summary}",
                metadata={
                    "source_type": "image_summary",
                    "file_name":   img_info["source_docx"],
                    "image_name":  img_info["image_name"]
                }
            )
            image_documents.append(img_doc)
            print(f"   ✅ Image summarized: {img_info['image_name']}")
            print(f"      Preview: {summary[:150]}...")
        except Exception as e:
            print(f"   ⚠️  Could not summarize {img_info['image_name']}: {e}")

# Add image summaries to the main document pool
all_documents.extend(image_documents)


🔍 Scanning history_notes.docx for embedded images...
   🖼️  Found 2 image(s). Sending to GPT-4o Vision...
   ✅ Image summarized: history_notes_image1.png
      Preview: This image is a timeline representing significant events in Indian history related to the freedom struggle. Here's a detailed description:

1. **Conce...
   ✅ Image summarized: history_notes_image2.png
      Preview: It seems like the image you provided is not displaying any educational content or specific details from the history notes. It appears to be a simple b...


In [26]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter

In [27]:
# ── Text Splitter Configuration ───────────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,         # Each chunk ≈ 500 characters
    chunk_overlap=100,      # 100 char overlap between consecutive chunks
    length_function=len,
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""]  # Split on natural boundaries
)

In [28]:
split_docs = text_splitter.split_documents(all_documents)

In [29]:
print(f"📊 Original documents: {len(all_documents)}")
print(f"📊 After splitting:    {len(split_docs)} chunks")

📊 Original documents: 19
📊 After splitting:    96 chunks


In [30]:
print(split_docs[0])

page_content='173
 Introduction 
Plants exhibits varying degrees of 
organization. Atoms are organized into 
molecules, molecules into organelles, organelles 
into cells, cells into tissues and tissues into 
organs. The first account of internal structure 
of plants was published by English Physician 
Nehemiah Grew. He is known as Father of 
Plant Anatomy. Plant anatomy (Gk Ana = as 
under; T emnein = to cut) is the study of internal 
structure of plants. Y ou have already studied the' metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2019-04-01T17:20:00+05:30', 'author': "BYJU'S", 'keywords': 'Tamilnadu Board Class 10 Science Chapter 12', 'moddate': '2019-04-01T17:20:30+05:30', 'subject': 'Tamilnadu Board Class 10 Science Chapter 12', 'title': 'Tamilnadu Board Class 10 Science Chapter 12', 'source': 'source_docs\\science_chapter3.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_type': 'pdf', 'file_name': 'scie

In [31]:
# ── Stats by source type ────────────────────────────────────────────────
source_counts = {}
for doc in split_docs:
    st = doc.metadata.get("source_type", "unknown")
    source_counts[st] = source_counts.get(st, 0) + 1

In [32]:
print("📊 Chunks by source type:")
for src, count in source_counts.items():
    print(f"   {src:20s}: {count} chunks")

📊 Chunks by source type:
   pdf                 : 79 chunks
   docx                : 4 chunks
   html                : 3 chunks
   image_summary       : 10 chunks


In [33]:
print("🔍 Sample Chunk:")
print("-" * 60)
print(f"Metadata: {split_docs[5].metadata}")
print(f"Content : {split_docs[5].page_content}")
print("-" * 60)

🔍 Sample Chunk:
------------------------------------------------------------
Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2019-04-01T17:20:00+05:30', 'author': "BYJU'S", 'keywords': 'Tamilnadu Board Class 10 Science Chapter 12', 'moddate': '2019-04-01T17:20:30+05:30', 'subject': 'Tamilnadu Board Class 10 Science Chapter 12', 'title': 'Tamilnadu Board Class 10 Science Chapter 12', 'source': 'source_docs\\science_chapter3.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_type': 'pdf', 'file_name': 'science_chapter3.pdf'}
Content : 174
10th Standard Science
Cuticle is present on the outer wall of epidermis to 
check evaporation of water. Trichomes and root 
hairs are the epidermal outgrowths. 
Functions:
i) Epidermis protects the inner tissues.
ii) Stomata helps in transpiration.
iii)  Root hairs help in absorption of water and
minerals.
12.2.2 Ground Tissue System
It includes all the tissues of the plant 
bo

In [34]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [35]:
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY
)

In [36]:
# ── Build FAISS Index ────────────────────────────────────────────────────
vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings_model
)

In [37]:
# ── Persist to disk (so we don't rebuild every session) ───────────────────
vectorstore.save_local(str(FAISS_INDEX_PATH))
print(f"✅ FAISS vector store created and saved to: {FAISS_INDEX_PATH}/")
print(f"   Index files: {list(FAISS_INDEX_PATH.iterdir())}")

✅ FAISS vector store created and saved to: faiss_index/
   Index files: [WindowsPath('faiss_index/index.faiss'), WindowsPath('faiss_index/index.pkl')]


In [38]:
# ── Test Retrieval ────────────────────────────────────────────────────────
test_query = "What is the equation of photosynthesis?"
retriever  = vectorstore.as_retriever(search_kwargs={"k": 3})
test_docs  = retriever.invoke(test_query)

In [39]:
print(test_docs)

[Document(id='0e657242-e23f-41d9-90fd-bc62417fe313', metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2019-04-01T17:20:00+05:30', 'author': "BYJU'S", 'keywords': 'Tamilnadu Board Class 10 Science Chapter 12', 'moddate': '2019-04-01T17:20:30+05:30', 'subject': 'Tamilnadu Board Class 10 Science Chapter 12', 'title': 'Tamilnadu Board Class 10 Science Chapter 12', 'source': 'source_docs\\science_chapter3.pdf', 'total_pages': 14, 'page': 8, 'page_label': '9', 'source_type': 'pdf', 'file_name': 'science_chapter3.pdf'}, page_content='181\nPlant Anatomy and Plant Physiology\n12.9.3 Photosynthesis\nPhotosynthesis (Photo \n= light; synthesis = to \nbuild) is a process by which \nautotrophic organisms like \ngreen plants, algae and \nchlorophyll containing \nbacteria utilize the energy from sunlight to \nsynthesize their own food. In this process, \ncarbon dioxide combines with water in the \npresence of sunlight and chlorophyll to fo

In [43]:
import gradio as gr
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate

In [41]:

GRADIO_SYSTEM_PROMPT = """You are an expert and friendly tutor for 10th Standard Tamil Nadu State Board students.
You help students prepare for their Samacheer Kalvi board exams.

Use ONLY the information from the provided context (study materials uploaded by the teacher).
If the answer is not in the context, say:
"📚 This specific topic is not in the loaded study materials. Please refer to your Samacheer Kalvi textbook or ask your teacher."

Formatting Guidelines:
- Use simple language for a 15-year-old student
- Use bullet points for lists
- For formulas, present them clearly on a new line
- For History: mention dates and key people
- For Science: include the chemical equation or formula if relevant
- End with: 💡 Tip: [one exam tip related to this topic]

Context from study materials:
{context}

Chat History:
{chat_history}

Student Question: {question}

Answer:"""

In [44]:
gradio_prompt = PromptTemplate(
    input_variables=["context", "chat_history", "question"],
    template=GRADIO_SYSTEM_PROMPT
)

In [45]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

C:\Users\manik\AppData\Local\Temp\ipykernel_24132\2971518456.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(


In [47]:
# ── Custom System Prompt for 10th Std Tamil State Board ─────────────────
SYSTEM_PROMPT = """You are an expert tutor for 10th Standard Tamil Nadu State Board students.
You help students prepare for their board exams in a friendly, encouraging way.

Use ONLY the information from the provided context (textbooks and study materials) to answer.
If the answer is not in the context, say: "This topic is not in the loaded study materials. 
Please ask your teacher or refer to your Samacheer Kalvi textbook."

Guidelines:
- Use simple language suitable for a 15-year-old student
- For Science/Math: include formulas and equations clearly
- For History/Social: include dates and key figures
- For Tamil Literature: include author names and key works
- At the end of each answer, suggest 1 related follow-up question the student might want to ask
- Keep answers focused and exam-oriented

Context from study materials:
{context}

Chat History:
{chat_history}

Student Question: {question}

Answer:"""

In [51]:
gradio_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

In [52]:
gradio_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    ),
    memory=gradio_memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": gradio_prompt},
    verbose=False
)

In [53]:

def chat_with_bot(user_message: str, history: list):

    if history is None:
        history = []

    try:
        result = gradio_chain.invoke({"question": user_message})
        answer = result["answer"]

        # Add sources
        sources = result.get("source_documents", [])
        if sources:
            seen_sources = set()
            source_lines = []

            for doc in sources:
                fname = doc.metadata.get("file_name", "unknown")
                stype = doc.metadata.get("source_type", "?")

                label = f"{fname} [{stype}]"

                if label not in seen_sources:
                    source_lines.append(f"📄 {label}")
                    seen_sources.add(label)

            if source_lines:
                answer += "\n\n---\n**📚 Sources:**\n" + "\n".join(source_lines)

        # ✅ NEW MESSAGE FORMAT
        history.append({
            "role": "user",
            "content": user_message
        })

        history.append({
            "role": "assistant",
            "content": answer
        })

        return history

    except Exception as e:

        history.append({
            "role": "assistant",
            "content": f"⚠️ Error: {str(e)}"
        })

        return history


In [54]:
def clear_chat():
    """Clear conversation history and memory."""
    gradio_memory.clear()
    return []

In [55]:

# ── Gradio Interface ───────────────────────────────────────────────────────
SAMPLE_QUESTIONS = [
    "What is photosynthesis? Explain the two stages.",
    "List the organs of the digestive system and their functions.",
    "What is the formula for nth term of AP? Solve an example.",
    "Explain the causes of the French Revolution.",
    "Who wrote Thirukkural? What are its three sections?",
    "What are the trigonometric values for 30°, 45°, and 60°?",
    "What is the discriminant and when does a quadratic have no real roots?",
    "Explain the First Anglo-Mysore War and its outcome.",
]

with gr.Blocks(
    title="📚 10th Std Tamil State Board - Exam Prep Bot",
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="green"),
    css="""
        .gradio-container { max-width: 900px !important; margin: auto; }
        #chatbot { height: 450px; }
        .sample-btn { font-size: 12px !important; }
    """
) as demo:
    
    # ── Header ──
    gr.HTML("""
    <div style='text-align:center; padding:20px; background: linear-gradient(135deg, #1a5276, #2e86c1); 
                border-radius:12px; margin-bottom:16px; color:white;'>
        <h1 style='margin:0; font-size:28px;'>📚 Tamil Nadu State Board</h1>
        <h2 style='margin:4px 0; font-size:20px; font-weight:400;'>10th Standard AI Exam Prep Bot</h2>
        <p style='margin:8px 0 0 0; opacity:0.85; font-size:14px;'>
            Powered by LangChain + OpenAI GPT-4o + FAISS | Based on Samacheer Kalvi Materials
        </p>
    </div>
    """)
    
    # ── Info Row ──
    with gr.Row():
        gr.HTML("""
        <div style='background:#eaf4fb; border-left:4px solid #2e86c1; padding:12px; border-radius:6px; font-size:13px;'>
            <b>📖 Loaded Study Materials:</b> Science (Photosynthesis, Digestive System) |
            Social Science (French Revolution) | Maths (AP, Trigonometry, Coordinate Geometry) |
            Tamil Literature (Thirukkural) | History (Anglo-Mysore War)
            <br><b>🧠 Knowledge Base:</b> FAISS vector store with OpenAI embeddings
        </div>
        """)
    
    gr.HTML("<div style='margin: 8px 0;'></div>")
    
    # ── Chat Interface ──
    chatbot = gr.Chatbot(
        elem_id="chatbot",
        label="Chat with your Study Bot",
        avatar_images=(
            "https://api.dicebear.com/7.x/pixel-art/svg?seed=student",
            "https://api.dicebear.com/7.x/bottts/svg?seed=studybot"
        )
    )
    
    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Ask your exam question here... e.g., 'What is photosynthesis?'",
            label="Your Question",
            lines=2,
            scale=5
        )
        with gr.Column(scale=1, min_width=120):
            submit_btn = gr.Button("Ask 🚀", variant="primary", size="lg")
            clear_btn  = gr.Button("Clear 🗑️", variant="secondary", size="sm")
    
    # ── Sample Questions ──
    gr.HTML("<p style='font-weight:bold; margin: 8px 0 4px 0; color:#555;'>💡 Try these sample questions:</p>")
    with gr.Row():
        for q in SAMPLE_QUESTIONS[:4]:
            sample_btn = gr.Button(q[:50] + "...", size="sm", elem_classes="sample-btn")
            sample_btn.click(
                fn=lambda q=q, hist=chatbot: chat_with_bot(q, hist or []),
                inputs=[chatbot],
                outputs=[chatbot]
            )
    with gr.Row():
        for q in SAMPLE_QUESTIONS[4:]:
            sample_btn = gr.Button(q[:50] + "...", size="sm", elem_classes="sample-btn")
            sample_btn.click(
                fn=lambda q=q, hist=chatbot: chat_with_bot(q, hist or []),
                inputs=[chatbot],
                outputs=[chatbot]
            )
    
    # ── Footer ──
    gr.HTML("""
    <div style='text-align:center; margin-top:16px; font-size:12px; color:#888;'>
        🔒 Answers are based only on your loaded study materials | 
        Built with LangChain 1.3.0 + langchain-openai 1.2.1 + FAISS + Gradio
    </div>
    """)
    
    # ── Event Handlers ──
    submit_btn.click(
        fn=chat_with_bot,
        inputs=[user_input, chatbot],
        outputs=[chatbot]
    ).then(lambda: "", outputs=[user_input])
    
    user_input.submit(
        fn=chat_with_bot,
        inputs=[user_input, chatbot],
        outputs=[chatbot]
    ).then(lambda: "", outputs=[user_input])
    
    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot]
    )


# ── Launch ────────────────────────────────────────────────────────────────
print("🚀 Launching Gradio Exam Prep Bot...")
demo.launch(
    share=False,          # Set share=True to get a public URL for sharing with students
    server_port=7868,
    inbrowser=True        # Auto-opens in browser
)

C:\Users\manik\AppData\Local\Temp\ipykernel_24132\2655840594.py:13: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


🚀 Launching Gradio Exam Prep Bot...
* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
